# 06: 多方法注释与交叉比对

对每个 Leiden 簇并行运行多种标注方法，交叉比对结果。
**多方法并跑不是冗余——不同独立方法的共识最能提高置信度**，
分歧标记需要 PI 重点复核的簇。

本 notebook 产出：
- 各方法独立标注列（`cell_type_{method}_v1`）
- 成对混淆矩阵热图 + Cohen's kappa 表
- 每簇 LLM 综合判决 markdown（需配 API key）
- PI 最终标注列 `cell_type_final_v1`

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：05（多分辨率 Leiden 聚类），读 `05_clustered_v*.h5ad`
- **下游**：06c（亚群 Subset 重分析）/ 07（下游分析），产出 `06_annotated_v*.h5ad`

### 为什么要迭代回跑？
注释质量直接影响下游逐簇分析和亚群重分析的准确性。如果在 06c（亚群分析发现注释不合理）、
07（跨病种比较时发现标签粒度不对）或逐簇报告中发现问题，可能需要：
- 换用不同的 Leiden 分辨率的列（修改 `LEIDEN_COL`）
- 增加或替换标记物 CSV（修改 `MARKER_CSV`）
- 调整 LLM 模型选择（修改 `MLLM_MODELS` / `MLLM_CONSENSUS_THRESHOLD` / `VERDICT_MODEL_TIER`）
- 在 PI 手动标注区修改 `marker_assignments` 或 `pi_decisions` 的标签
- 换用 05 的另一个版本（不同 Leiden 分辨率组合）

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `05_clustered_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `06_annotated_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `LEIDEN_COL`、`MARKER_CSV`、
   `LLM_MODELS` 等），然后 Run All 重跑全部 cell。

> 原始构思（PI）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建、
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

**起点衔接——本 notebook 默认从 05 的最稳定版本出发。**

In [ ]:
# === PARAMS ===
# UPSTREAM_RUN_ROOT / UPSTREAM_RUN_ID — 已提升的 Stage 05 run
# RUN_ROOT / RUN_ID / OUTPUT_FILENAME — 本次 Stage 06 draft
# MARKER_CSV     — 标记物知识库 CSV（供 dotplot + 基因集评分）
# LEIDEN_COL     — 用作簇标签的 obs 列
# RANDOM_SEED    — 随机种子，确保可复现

UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUN_ID = "05-selected-run"
RUN_ROOT = "results/runs"
RUN_ID = "06-annotation-run"
OUTPUT_FILENAME = "06_annotated.h5ad"

MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"  # 测试夹具，PI 后续替换为真实 marker 库
LEIDEN_COL  = "leiden_res_0.6"
RANDOM_SEED = 42

# 基因集评分最低阈值——argmax 取最高分细胞类型时，低于此阈值的细胞标为 "Unknown"
# 默认 0 = 不启用过滤（保持旧行为）；建议设为正值（如 0.05）以过滤噪声
SCORE_MIN_THRESHOLD = 0

# === 注释产物版本（决策6：版本解耦，单一版本号拆为两个） ===
# ANNOTATION_OUTPUT_VERSION — 本 notebook 注释产出版本号
#   所有 obs 列名 / uns key 的后缀由它单点生成（如 cell_type_marker_v1、annotation_provenance_v1）
#   默认 "v1" 与现有下游（07）的读取契约兼容；Bump 此处即可统一更新所有注释相关列名
# UPSTREAM_LABEL_VERSION — 上游粗标签版本（若 05 带版本的 label 列）
#   P3-b 仅登记 + 轻度使用；P3-c 与 MAIN/SUBSET 完全解耦时收口
ANNOTATION_OUTPUT_VERSION = "v1"  # 注释产出版本（替代旧 OUTPUT_VERSION）
UPSTREAM_LABEL_VERSION = "v1"     # 上游标签版本（P3-c 收口）

# === PI 确认闸门控制（决策6：机器建议不自动落地，PI 必须显式确认） ===
# 为什么需要 PI_CONFIRMED 闸门？GREEN 只是 LLM 的机器置信度高，
# 不等于 PI 已经判断其生物学合理性。final cell type 涉及下游分析的全部解释，
# 必须由 PI 逐簇确认后才能写入。在 PI_CONFIRMED=False 时，
# 本 notebook 只产出 suggested 层（机器建议）和 pi_confirmed 层（PI 决定），
# 绝不写 cell_type_final 列。
PI_CONFIRMED = False  # PI 显式确认闸门总开关；False 时永远只写 draft、不写 final
ACCEPT_ALL_SUGGESTED = False  # "全部接受建议"批量操作；仅当 PI_CONFIRMED=True 且本值 True 时，
                              # 才从 suggested 生成逐簇 accept 审计记录（一次显式确认）
final_gate_passed = False  # 显式闸门结果标记变量，由 gate cell 赋值；
                           # checkpoint cell 读它决定 NEEDS_REVIEW vs SUCCESS
                           # PARAMS 先给确定初值。禁的是 dir() 流控残留（靠 if x in dir() 决定业务分支）；防御性存在性检查允许

# === PI 确认输入方式（扩展为新 6 列 schema，覆盖全部 cluster） ===
# 支持两种方式：
#   方式 1（推荐非 CS 用户）：CSV 文件，6 列
#     cluster, suggested_label, pi_final_label, decision(accept|modify|unresolved), note, annotation_version
#     在 PI_CONFIRMATION_CSV 中设文件路径，然后在 Excel/WPS 中编辑即可
#   方式 2（程序员）：直接编辑下方 pi_decisions Python 字典
# 向后兼容：仅含 cluster, label 两列的旧 CSV 视为 decision=modify（PI 手填即明确决定）
PI_CONFIRMATION_CSV = ""  # 留空则使用 Python dict；填路径则从 CSV 读取
                          # CSV 格式: cluster,suggested_label,pi_final_label,decision,note,annotation_version

# === LLM 配置（读 vault 根 .env 的 LLM_GROUP{N}_* schema）===
# LLM_GROUP  — 使用第几个 LLM group（对应 .env 的 LLM_GROUP{N}_*）
#              None = 自动用 LLM_DEFAULT_GROUP（当前 .env 里设为 1）
#              每个 group 包含 1 个 provider + base_url + api_key + 3 档模型
#              多 group 可用时，mLLMCelltype 共识会自动使用多个端点
LLM_GROUP = None  # None → 自动取 LLM_DEFAULT_GROUP；手工指定：LLM_GROUP = 1

# --- LLM 注释（mLLMCelltype 多模型共识）---
# mLLMCelltype 同时调用多个 LLM 模型独立注释每簇，然后多数票投票产生共识标签
# 当模型间分歧 > 阈值时自动触发多轮讨论（discussion rounds），直到达成一致
MLLM_ENABLED = True
MLLM_MODELS = None          # None → 从 .env LLM_GROUP 自动构建模型列表
                             # 或手动指定如 ["claude-3-5-haiku-20241022", "deepseek-chat"]
MLLM_CONSENSUS_THRESHOLD = 0.7   # 标签比例需 >=70% 才算"达成共识"
MLLM_ENTROPY_THRESHOLD = 0.3     # 标签分布熵需 <=0.3 才算"意见集中"
MLLM_MAX_DISCUSSION_ROUNDS = 2   # 多模型讨论最多 2 轮

# --- LLM 证据汇总判决 ---
#   "sonnet" — 平衡，适合证据汇总判决（推理-成本平衡）
#   "opus"  — 最强推理，适合争议簇复核（贵、慢，按需启用）
VERDICT_MODEL_TIER = "sonnet"  # 证据汇总用
# 思考模型（如 deepseek-v4-pro）单次输出可达 ~7k token，800 会截断 JSON 致 JSONDecodeError
# 若切换非思考模型（如 gpt-4o-mini）可降低此值
LLM_MAX_TOKENS = 16384

# === scANVI 参考 atlas（不存在则优雅跳过） ===
REFERENCE_ATLAS_PATH = ""  # 留空跳过；填入 .h5ad 路径启用


In [ ]:
# === setup：sys.path + 导入 + 加载上游 adata + 标记物知识库 ===

# 1. 确保框架 src/ 在 sys.path 上，CWD 为项目根目录
import sys, os, gc
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/06_verdicts", exist_ok=True)
os.makedirs("results/figures/06_sankey", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 2. 导入（scanpy 原生 API + 框架函数）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import datetime, json, re, warnings
from pathlib import Path
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    resume_run, sha256_file, snapshot_effective_parameters, validate_checkpoint,
)

# 标记物库加载器（替代原来的 from scrna_integration import load_markers）
# src/notebook 边界铁律（2026-07-10 PI 定稿）：分析逻辑留 notebook cell，可见可调可开关
def load_markers(csv_path, roles=("canonical", "optional")):
    """加载标记物 CSV，按 cell_type 分组并按 role 过滤。

    CSV 格式: tissue, cell_type, marker, role, reference, notes
    Args:
        csv_path: CSV 文件路径
        roles: role 过滤元组，默认 ("canonical", "optional")；None 返回完整三层字典
    Returns: 细胞类型 → 标记物列表（扁平）或 细胞类型 → {role: [...]}（嵌套）
    """
    df = pd.read_csv(csv_path, comment="#")
    _required = ["cell_type", "marker", "role"]
    _missing = [col for col in _required if col not in df.columns]
    if _missing:
        raise ValueError(
            f"marker CSV 缺少必需列: {', '.join(_missing)}；"
            f"需要的列: {', '.join(_required)}。"
            f"当前文件列: {', '.join(df.columns.tolist())}。"
            f"请检查 CSV 列名是否与模板一致（区分大小写）。"
        )
    if isinstance(roles, str):
        raise TypeError(
            "roles 参数必须为 list 或 tuple，不能传字符串。"
            "若只想查一种 role，请写成 ('canonical',) 注意末尾逗号。"
        )
    if roles is None:
        result = {}
        for role in ("canonical", "optional", "negative"):
            role_df = df[df["role"] == role]
            for ct, group in role_df.groupby("cell_type"):
                result.setdefault(ct, {}).setdefault(role, [])
                result[ct][role] = group["marker"].tolist()
        for ct in result:
            for role in ("canonical", "optional", "negative"):
                result[ct].setdefault(role, [])
        return result
    filtered = df[df["role"].isin(roles)]
    return {ct: group["marker"].tolist() for ct, group in filtered.groupby("cell_type")}
# 标注一致性评分（替代原来的 from scrna_integration.scorers import annotation_concordance）
# src/notebook 边界铁律（2026-07-10 PI 定稿）：分析逻辑留 notebook cell，可见可调可开关
from sklearn.metrics import cohen_kappa_score

def _first_match(columns, candidates):
    """返回 columns 中第一个名称包含 candidates 中任一项的列。"""
    for candidate in candidates:
        for col in columns:
            if candidate in col:
                return col
    return None

def annotation_concordance(adata, label_a=None, label_b=None):
    """两个标注列之间的 Cohen's kappa 一致性系数。

    自动检测 obs 中包含 cell_type/label/annotation/leiden 的列作为两个标注列。
    Returns: {"cohen_kappa": float} 或 {"_note": NaN}
    """
    if label_a is None or label_b is None:
        candidates = [
            c for c in adata.obs.columns
            if any(p in c.lower() for p in ("cell_type", "label", "annotation", "leiden"))
        ]
        if label_a is not None:
            candidates = [c for c in candidates if c != label_a]
        elif label_b is not None:
            candidates = [c for c in candidates if c != label_b]
        if len(candidates) >= 2 and label_a is None:
            label_a = candidates[0]
        if len(candidates) >= 2 and label_b is None:
            label_b = candidates[1]
    if (label_a is None or label_b is None or label_a == label_b
        or label_a not in adata.obs.columns or label_b not in adata.obs.columns):
        return {"_note": float("nan")}
    a = adata.obs[label_a].astype(str)
    b = adata.obs[label_b].astype(str)
    valid = adata.obs[label_a].notna() & adata.obs[label_b].notna()
    return {"cohen_kappa": float(cohen_kappa_score(a[valid], b[valid]))}

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = "results/figures"
warnings.filterwarnings("ignore", category=FutureWarning)


# 安全排序 key：处理簇 ID 可能为非数值字符串（如 "Chief cell"）
def _safe_sort_key(x):
    try:
        return (0, int(x))
    except (ValueError, TypeError):
        return (1, str(x))

# === 只读取 verified promoted Stage 05 上游 ===
upstream_run = resume_run(UPSTREAM_RUN_ROOT, UPSTREAM_RUN_ID, promoted=True)
upstream_manifest_path = upstream_run.promoted_dir / "manifest.json"
upstream_manifest = json.loads(upstream_manifest_path.read_text(encoding="utf-8"))
if upstream_manifest.get("run_id") != UPSTREAM_RUN_ID:
    raise ValueError("upstream manifest run_id 与配置不匹配")
if upstream_manifest.get("stage") != "05_clustered":
    raise ValueError("upstream manifest stage 必须是 05_clustered")
upstream_stage_status = upstream_manifest.get("stage_status")
if upstream_stage_status == "SUCCESS_WITH_WARNINGS":
    warning_acceptance = upstream_manifest.get("warning_acceptance")
    if not isinstance(warning_acceptance, dict) or not all(
        isinstance(warning_acceptance.get(field), str) and warning_acceptance[field].strip()
        for field in ("accepted_by", "accepted_at")
    ):
        raise ValueError("SUCCESS_WITH_WARNINGS 上游缺少有效 warning_acceptance")
elif upstream_stage_status != "SUCCESS":
    raise ValueError("upstream manifest stage_status 不可供 Stage 06 消费")
UPSTREAM_CHECKPOINT = validate_checkpoint(upstream_manifest_path)
upstream_input = {
    "run_id": UPSTREAM_RUN_ID, "stage": "05_clustered",
    "manifest_path": str(upstream_manifest_path),
    "manifest_sha256": sha256_file(upstream_manifest_path),
    "checkpoint_path": str(UPSTREAM_CHECKPOINT),
    "checkpoint_sha256": sha256_file(UPSTREAM_CHECKPOINT),
}
print("加载上游:", UPSTREAM_CHECKPOINT)
adata = sc.read_h5ad(UPSTREAM_CHECKPOINT)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"obsm 键: {list(adata.obsm.keys())}")
print(f"leiden 列 '{LEIDEN_COL}': "
      f"{adata.obs[LEIDEN_COL].nunique()} 个簇" if LEIDEN_COL in adata.obs else "缺失")
# === promoted Stage 05 上游读取完成 ===
# === Stage 06 input preflight：任何注释方法消费 LEIDEN_COL 之前 ===
input_hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "cluster_column_present": LEIDEN_COL in adata.obs.columns,
    "cluster_labels_complete": LEIDEN_COL in adata.obs and bool(adata.obs[LEIDEN_COL].notna().all()),
    "cluster_labels_nonempty": LEIDEN_COL in adata.obs and bool(adata.obs[LEIDEN_COL].dropna().astype(str).str.strip().ne("").all()),
}
if not all(input_hard_postconditions.values()):
    run_paths = prepare_run(RUN_ROOT, RUN_ID)
    manifest_payload = {
        "run_id": RUN_ID, "stage": "06_annotated", "stage_status": "FAILED",
        "inputs": [upstream_input], "effective_parameters": snapshot_effective_parameters(globals(), exclude=("UPSTREAM_CHECKPOINT",), path_root=Path(_root)),
        "runtime_provenance": collect_runtime_provenance(_root, ("anndata", "scanpy", "numpy", "pandas", "scipy")),
        "hard_postconditions": input_hard_postconditions, "failure": {"type": "InputContractError", "message": "LEIDEN_COL preflight failed"},
    }
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 06 input preflight FAILED: {input_hard_postconditions}")
# === Stage 06 input preflight 完成 ===

# 4. 加载标记物知识库（marker CSV 不存在时优雅跳过）
if os.path.exists(MARKER_CSV):
    markers = load_markers(MARKER_CSV)
    print(f"标记物库: {MARKER_CSV}")
    print(f"  细胞类型数: {len(markers)}")
    for ct, genes in list(markers.items())[:5]:
        print(f"  {ct}: {genes}")
    if len(markers) > 5:
        print(f"  ... 共 {len(markers)} 种细胞类型")

    # 展平为所有标记基因列表（用于 dotplot）
    all_marker_genes = sorted(set(g for glist in markers.values() for g in glist))
    # 只保留在 adata 中实际存在的基因
    available_markers = [g for g in all_marker_genes if g in adata.var_names]
    missing = set(all_marker_genes) - set(available_markers)
    if missing:
        print(f"  数据中不存在的标记基因（跳过）: {sorted(missing)}")
    print(f"  可用标记基因: {len(available_markers)}/{len(all_marker_genes)}")
else:
    print(f"marker CSV 不存在 ({MARKER_CSV})，跳过标记物知识库加载")
    markers = {}
    available_markers = []

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包，见 ADR-0012）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}")

In [ ]:
# === LLM 响应解析（从 llm_config 导入健壮函数） ===
# 原内联 helper 已迁入 src/scrna_integration/llm_config.py（批 5 收口）。
# notebook 里一行导入即可，非 CS 学生不需关心 JSON 格式处理细节。
from scrna_integration.llm_config import extract_json_from_llm_response


## 方法 1：标记物 dotplot（PI 手动标注）

用已有标记物知识库画 dotplot，每个簇表达哪些标记物一目了然。
**为什么先做这个？** 让 PI 在受自动方法影响前建立自己的判断，避免锚定偏差。
（也可以后做——方法顺序不影响结果，PI 自由选择。）

### 怎么看 dotplot？
- **横轴**：标记基因；**纵轴**：簇
- **颜色深浅**：该基因在该簇的平均表达量
- **圆点大小**：该簇中表达该基因的细胞百分比
- **判断规则**：某个簇对某类细胞的全部标记基因都表达（大圆点 + 深色）
  → 该簇很可能是该细胞类型；只表达个别标记基因 → 可能不是或需更多证据

PI 浏览此图后在下方的 `marker_assignments` 字典中填写每个簇的细胞类型。

In [ ]:
# 标记物 dotplot——每个簇 x 每个标记基因的（表达百分比 + 平均表达量）
if available_markers and LEIDEN_COL in adata.obs.columns:
    sc.pl.dotplot(
        adata, var_names=available_markers, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot ({LEIDEN_COL})",
        save="_06_dotplot.png",
    )
    plt.close("all")
else:
    print("无可用的标记基因或缺少 leiden 列，跳过 dotplot")

In [ ]:
# === Marker Dotplot（按细胞类型分组）===
# 每个细胞类型作为一组（而非把所有 marker 混在一起），
# 组间用分隔线区分，更直观地展示"该簇是否符合某细胞类型的完整 marker profile"。
# 为什么保留 flat dotplot（上方 cell）？flat 版本展示全部 marker 的相对排序，
# 分组版本展示类型内 marker 一致性——两个视角互补。

if markers and available_markers:
    # 构建分组 dict 供 sc.pl.dotplot 使用
    _var_names_grouped = {}
    for ctype, genes in markers.items():
        _present = [g for g in genes if g in adata.var_names]
        if _present:
            _var_names_grouped[ctype] = _present

    if _var_names_grouped:
        print(f"Dotplot 分组: {len(_var_names_grouped)} 组, 共 {len(available_markers)} 个标记基因")
        sc.pl.dotplot(
            adata,
            var_names=_var_names_grouped,
            groupby=LEIDEN_COL,
            standard_scale="var",
            dendrogram=True,
            show=True,
            save="_06_dotplot_grouped.png",
        )
        plt.close("all")
    else:
        print("⚠️ 无可用标记基因，跳过分组 dotplot")


In [ ]:
# === 基于 dotplot 数据的自动标签建议 ===
# 对每个 cluster，找表达最高的 marker 类别 → 建议标签
if markers and LEIDEN_COL in adata.obs.columns:
    print("===== 基于 marker 表达的自动标签建议 =====\n")
    _suggestions = {}
    
    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=lambda x: int(x) if x.isdigit() else x):
        _mask = adata.obs[LEIDEN_COL] == cl
        _best_score = -1
        _best_type = "Unknown"
        
        for ctype, genes in markers.items():
            _present = [g for g in genes if g in adata.var_names]
            if not _present:
                continue
            _expr = adata[_mask][:, _present].X
            if sp.issparse(_expr):
                _expr = _expr.toarray()
            _mean_expr = _expr.mean()
            if _mean_expr > _best_score:
                _best_score = _mean_expr
                _best_type = ctype
        
        _suggestions[str(cl)] = _best_type
        print(f"  Cluster {cl}: → {_best_type} (mean expr={_best_score:.3f})")
    
    print(f"\n  ⚠️ 以上为纯计算建议（marker 库最高均值表达匹配），仅做参考")
    print(f"  → PI 应结合 dotplot 视觉判断和 LLM 证据综合决策")
    print(f"\n  如接受上述建议，可直接复制到 marker_assignments:")
    print(f"  marker_assignments = {_suggestions}")

In [ ]:
# === PI 手动标注区（方法 1：marker 注释）===
# PI 查看上方 dotplot 后，在下方字典中为每个簇 ID 填入细胞类型。
# 示例（基于 Nowicki 数据——PI 需根据实际 dotplot 修改）：
marker_assignments = {
    # "0": "B_cell",
    # "1": "T_cell",
    # "2": "myeloid",
    # ...  PI 逐簇填入
}

# B2 开关式裁剪：只有 PI 实际填写了 marker_assignments 才创建标注列。
# 空 dict 不创建空列（不再是空 dict 假装跑了）。
_marker_col = f"cell_type_marker_{ANNOTATION_OUTPUT_VERSION}"
if marker_assignments:
    adata.obs[_marker_col] = (
        adata.obs[LEIDEN_COL].astype(str).map(marker_assignments)
    ).astype("category")
    n_assigned = adata.obs[_marker_col].notna().sum()
    print(f"marker 标注: {n_assigned}/{adata.n_obs} 细胞已标注 "
          f"({len(marker_assignments)} 簇) -> obs['{_marker_col}']")
else:
    # PI 暂未填写——不创建空列，交叉比对时不纳入
    print("marker 标注: PI 暂未填写 marker_assignments，跳过（不创建空列）")
    print("  -> 交叉比对将在可用方法中排除 marker 列")
    print("  -> PI 可随时填入上方字典并重新运行本 cell")


## 方法 2：mLLMCelltype 多模型共识注释

用 mLLMCelltype 的 `interactive_consensus_annotation()` 同时调用多个 LLM 模型
独立注释每簇，取多数票作为共识标签。

**为什么用 mLLMCelltype？**
- **多模型共识**：不同模型的独立判断互相验证，共识标签置信度远高于单模型
- **自动讨论机制**：模型间意见分歧时自动触发多轮讨论（discussion rounds），
  模拟"专家会诊"——各模型说明自己的推理并参考他人观点，直到达成共识
- **配置从 .env 自动读取**：无需手动管理 api_key 列表，框架自动构建
- **组织上下文**：传入 `tissue_type="human gastric mucosa"` 让 LLM 结合胃粘膜背景知识

### 共识参数怎么调？
- `consensus_threshold=0.7`：某标签获得 >=70% 模型投票即认为"达成共识"
- `entropy_threshold=0.3`：标签分布熵 <=0.3（低熵 = 意见集中）
- `max_discussion_rounds=2`：最多 2 轮讨论，避免无限辩论
- 调低阈值 → 更宽松（更快但可能遗漏分歧）；调高 → 更严格（更慢但更可靠）

### 怎么看注释结果？
每簇输出模型投票分布和最终共识标签。各模型投票不一致的簇自动标记为
"争议簇"（low-confidence），供 PI 重点复核。

> 技术说明：为什么从 requests 直连改回 mLLMCelltype？
> 之前因本地网关 thinking 块不兼容从 mLLMCelltype 切换到 requests 直连。
> 现在切换回来是因为：(1) mLLMCelltype 的 interactive_consensus_annotation
> 提供多模型讨论机制，比手写多数票投票更稳健；(2) 模型管理和错误重试由包内部处理，
> notebook 代码更简洁；(3) 上游已修复 thinking 块兼容性。


In [ ]:
# === 方法 2: mLLMCelltype 多模型共识注释 ===
# 从 .env LLM_GROUP 自动构建模型列表 -> 每簇独立注释 -> 多模型投票共识
# 
# 兼容性说明：
# 本 cell 通过 llm_config.apply_mllmcelltype_patches() 修复 mLLMCelltype 库的已知缺陷
# （timeout 过小 / max_tokens 过低 / thinking 块解析不兼容）。
# 这些 patch 依赖库内部结构（PROVIDER_FUNCTIONS dict、
# _parse_anthropic_response 等），库大版本升级可能破坏兼容。
# 若 patch 失败，该 cell 优雅降级为跳过 mLLMCelltype 注释，
# 不影响其他 cell（verdict 三色判决路径独立、不依赖库 patch）。
# 已在 mllmcelltype==2.1.1 上验证通过（2026-06-21）。

if MLLM_ENABLED:
    try:
        from mllmcelltype import interactive_consensus_annotation
        
        # ---- 应用可读 monkey-patch（参数全部显式命名，非 CS 学生可看懂） ----
        from scrna_integration.llm_config import (
            apply_mllmcelltype_patches,
            build_mllmcelltype_config,
        )
        _patch_ok = apply_mllmcelltype_patches(
            max_retries=3,
            retry_delay=2,
            timeout=120,
            max_tokens_override=16384,
        )

        # ----------------------------------------------------------------
        if not _patch_ok:
            print("mLLMCelltype 库 patch 失败，跳过（见上方 warning 详情）")
        else:
            # ---- 从 .env 自动构建 mLLMCelltype 配置 ----
            _api_keys, _base_urls, _model_list = build_mllmcelltype_config(
                project_root=_root,
                model_list_override=MLLM_MODELS,
            )
            _n_models = len(_model_list) if _model_list else 0
            _use_discussion = _n_models >= 2
            if not _use_discussion:
                print(f"\u26a0\ufe0f 仅配置 {_n_models} 个模型，多模型共识降级为单模型注释")
                print(f"  -> 单一模型标签未经交叉验证，置信度低于多模型共识")
                print(f"  -> 如需多模型共识，请在 .env 中配置多个 LLM_GROUP* 或同一 group 的多档模型")
            if not _use_discussion:
                print(f"\u26a0\ufe0f 仅配置 {_n_models} 个模型，多模型共识降级为单模型注释")
                print(f"  -> 单一模型标签未经交叉验证，置信度低于多模型共识")
                print(f"  -> 如需多模型共识，请在 .env 中配置多个 LLM_GROUP* 或同一 group 的多档模型")
            if not _use_discussion:
                print(f"\u26a0\ufe0f 仅配置 {_n_models} 个模型，多模型共识降级为单模型注释")
                print(f"  -> 单一模型标签未经交叉验证，置信度低于多模型共识")
                print(f"  -> 如需多模型共识，请在 .env 中配置多个 LLM_GROUP* 或同一 group 的多档模型")
            
            if _model_list:
                print(f"mLLMCelltype 模型: {_model_list}")
                print(f"开始多模型共识注释（{_n_models} 模型，{'讨论模式' if _use_discussion else '直注模式'}）...")
                
                if "rank_genes_06" not in adata.uns:
                    sc.tl.rank_genes_groups(
                        adata, groupby=LEIDEN_COL, method="wilcoxon",
                        n_genes=30, key_added="rank_genes_06",
                    )
                
                _rgg = adata.uns["rank_genes_06"]
                _cluster_names = list(_rgg["names"].dtype.names)
                _marker_genes = {}
                for cl in _cluster_names:
                    _marker_genes[str(cl)] = _rgg["names"][cl][:10].tolist()
                
                _result = interactive_consensus_annotation(
                    marker_genes=_marker_genes,
                    species="human",
                    models=_model_list,
                    api_keys=_api_keys,
                    base_urls=_base_urls if _base_urls else None,
                    tissue="human gastric mucosa",
                    consensus_threshold=MLLM_CONSENSUS_THRESHOLD,
                    entropy_threshold=MLLM_ENTROPY_THRESHOLD,
                    max_discussion_rounds=(
                        MLLM_MAX_DISCUSSION_ROUNDS if _use_discussion else 0
                    ),
                    use_cache=False,
                )
                
                # 解析结果（兼容多种返回格式）
                if hasattr(_result, "cell_types") and _result.cell_types:
                    _llm_map = _result.cell_types
                    adata.obs[f"cell_type_llm_{ANNOTATION_OUTPUT_VERSION}"] = (
                        adata.obs[LEIDEN_COL].astype(str).map(_llm_map)
                    ).astype("category")
                    print(f"mLLMCelltype: {len(_llm_map)} 簇")
                elif isinstance(_result, dict) and "consensus" in _result:
                    _consensus = _result["consensus"]
                    adata.obs[f"cell_type_llm_{ANNOTATION_OUTPUT_VERSION}"] = (
                        adata.obs[LEIDEN_COL].astype(str).map(_consensus)
                    ).astype("category")
                    print(f"mLLMCelltype: {len(_consensus)} 簇")
                    for cid, label in sorted(
                        _consensus.items(),
                        key=lambda x: int(x[0]) if x[0].isdigit() else x[0],
                    ):
                        print(f"    Cluster {cid}: {label}")
                elif isinstance(_result, dict):
                    adata.obs[f"cell_type_llm_{ANNOTATION_OUTPUT_VERSION}"] = (
                        adata.obs[LEIDEN_COL].astype(str).map(_result)
                    ).astype("category")
                    print(f"mLLMCelltype (dict): {len(_result)} 簇")
                else:
                    print(f"mLLMCelltype 返回: {type(_result).__name__}")
            else:
                print("无可用模型，跳过 mLLMCelltype（.env 未配置 LLM_GROUP*）")
    except ImportError:
        print("mLLMCelltype 未安装，跳过")
    except Exception as e:
        print(f"mLLMCelltype 异常: {e}")
        import traceback
        traceback.print_exc()
        print("  -> fallback: verdict 三色判决路径独立，不受影响")
else:
    print("MLLM_ENABLED=False，跳过 mLLMCelltype")

if "cell_type_llm_v1" not in adata.obs.columns:
    adata.obs[f"cell_type_llm_{ANNOTATION_OUTPUT_VERSION}"] = pd.Categorical([np.nan] * adata.n_obs)


## 方法 3：基因集评分

用 `sc.tl.score_genes` 对每类标记基因集合做评分，得到每个细胞相对于每个
细胞类型的连续得分（`obs["score_{celltype}"]`）。

**为什么做基因集评分？** 评分提供了连续性证据——一个簇可能同时高表达多种
细胞类型的标记，说明该簇可能是过渡态或混合群体。评分不直接作为独立标签，而是
作为交叉比对的补充证据：当多个方法对同一簇的标签有分歧时，看该簇对哪种细胞类型的
评分更高，辅助裁决。

### 怎么看评分图？
下方 UMAP 图中，颜色深浅 = 该细胞对该细胞类型的基因集评分（0 到高值）。
某个簇整体颜色深 → 该簇高表达该类标记物 → 支持该细胞类型标注。

In [ ]:
# 对每个细胞类型做基因集评分（sc.tl.score_genes）
# 原理：标记基因平均表达 - 随机参考基因平均表达，产生连续得分
# 用 score_genes 而非 AUCell：scanpy 原生，零额外依赖，学生直接看懂

_score_cols = []
for ct, gene_list in markers.items():
    # 只保留数据中实际存在的基因
    _present = [g for g in gene_list if g in adata.var_names]
    if len(_present) < 2:
        print(f"  {ct}: 可用标记基因 <2（{len(_present)}），跳过评分")
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
    print(f"  {ct}: {len(_present)}/{len(gene_list)} 个基因可用 -> obs['{col}']")

print(f"\n共生成 {len(_score_cols)} 个评分列")

# B2：基因集评分完成 → 创建离散标签列（每细胞取最高评分的细胞类型）
# 纳入交叉比对，作为独立注释方法
# P2-2: argmax 加最低分阈值——低于 SCORE_MIN_THRESHOLD 的细胞标为 "Unknown"，
# 避免低置信度噪声拉低交叉比对 kappa。默认阈值 0 = 不启用过滤。
if _score_cols:
    _score_data = adata.obs[_score_cols].values
    _best_idx = np.argmax(_score_data, axis=1)
    _best_scores = np.max(_score_data, axis=1)
    _score_labels = [col.removeprefix("score_") for col in _score_cols]
    _best_labels = [_score_labels[i] for i in _best_idx]
    # 低于阈值的细胞标为 Unknown（阈值 > 0 时才生效，默认 0 保持旧行为）
    if SCORE_MIN_THRESHOLD > 0:
        _low_mask = _best_scores < SCORE_MIN_THRESHOLD
        _best_labels = [
            "Unknown" if _low_mask[i] else lbl
            for i, lbl in enumerate(_best_labels)
        ]
        n_unknown = int(_low_mask.sum())
        print(f"基因集评分阈值 {SCORE_MIN_THRESHOLD}: {n_unknown} 个细胞 (< 阈值) → Unknown")
    _scores_col = f"cell_type_scores_{ANNOTATION_OUTPUT_VERSION}"
    adata.obs[_scores_col] = pd.Categorical(_best_labels)
    print(f"基因集评分标签列已创建: obs['{_scores_col}'] "
          f"({adata.obs[_scores_col].nunique()} 类)")
else:
    print("基因集评分: 无评分列生成，跳过标签列创建")


In [ ]:
# 逐簇汇总基因集评分——均值 + 阳性细胞百分比
# 单个细胞的评分有噪声，簇级汇总能更稳健地反映该簇的整体标记物信号

if _score_cols and LEIDEN_COL in adata.obs.columns:
    _records = []
    if not hasattr(adata.obs[LEIDEN_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LEIDEN_COL]):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype("category")
    for _cid in sorted(adata.obs[LEIDEN_COL].cat.categories):
        _mask = adata.obs[LEIDEN_COL] == _cid
        _row = {"cluster": _cid, "n_cells": _mask.sum()}
        for col in _score_cols:
            _vals = adata.obs.loc[_mask, col]
            _ct_name = col.removeprefix("score_")
            _row[f"{_ct_name}_mean"] = round(float(_vals.mean()), 4)
            _row[f"{_ct_name}_pct_pos"] = round(float((_vals > 0).mean()) * 100, 1)
        _records.append(_row)
    _score_summary = pd.DataFrame(_records)
    print("基因集评分逐簇汇总:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(_score_summary)
    except ImportError:
        print(_score_summary.to_string())
    _score_summary.to_csv("results/tables/06_gene_set_scores.csv", index=False)
    print("\n已保存: results/tables/06_gene_set_scores.csv")
else:
    print("无评分列或缺少 leiden 列，跳过逐簇汇总")

In [ ]:
# 基因集评分可视化——UMAP 着色
# 每个子图对应一种细胞类型的评分，颜色深浅 = 该细胞对该类型的得分
# PI 借此判断哪些簇对哪种细胞类型的标记物评分最高

if _score_cols and "X_umap" in adata.obsm:
    n = len(_score_cols)
    n_cols = min(3, n)
    n_rows = (n + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5.5 * n_cols, 4.5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, col in enumerate(_score_cols):
        ct_name = col.removeprefix("score_")
        ax = axes[i]
        sc.pl.umap(adata, color=col, ax=ax, show=False,
                   title=ct_name, cmap="viridis",
                   vmin=0, vmax="p99", frameon=False)

    # 隐藏多余子图
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    fig.savefig("results/figures/06_gene_set_scores_umap.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("基因集评分 UMAP 已保存: results/figures/06_gene_set_scores_umap.png")
    plt.close("all")
elif not _score_cols:
    print("无评分列，跳过基因集评分可视化")
else:
    print("obsm 中无 X_umap，跳过 UMAP 着色")

## 方法 4：scANVI 标签迁移（守卫——有参考 atlas 时启用）

**前提**：存在对应疾病系统的有标注参考 atlas（含 `cell_type` obs 列）。
scANVI 同时利用参考数据的已知标签和目标数据自身的无监督结构做半监督训练，
标签迁移比简单 kNN 映射更稳健。

当前胃粘膜没有公认的标注参考，**自动跳过**。
后续若 CELLxGENE Census 或合作者提供有标注的胃参考数据，
在 PARAMS 中填入 `REFERENCE_ATLAS_PATH` 即可启用。

In [ ]:
# scANVI 标签迁移（守卫——REFERENCE_ATLAS_PATH 存在且有 cell_type 列时启用）
# 用 scvi-tools >= 1.0 API：参考+目标合并为单个 AnnData，
# 参考细胞的标签列保留原值，目标细胞标记为 "Unknown"。
# SCANVI 在半监督模式下同时利用参考标签和目标数据无监督结构做迁移。

_scANVI_col = f"cell_type_scanvi_{ANNOTATION_OUTPUT_VERSION}"
_scANVI_unc_col = f"cell_type_scanvi_{ANNOTATION_OUTPUT_VERSION}_uncertainty"

if REFERENCE_ATLAS_PATH and os.path.exists(REFERENCE_ATLAS_PATH):
    try:
        import scvi as _scvi

        _ref = sc.read_h5ad(REFERENCE_ATLAS_PATH)
        print(f"加载参考 atlas: {REFERENCE_ATLAS_PATH}")
        print(f"  参考: {_ref.n_obs:,} 细胞 x {_ref.n_vars:,} 基因")

        # 确认参考数据有 cell_type 列
        if "cell_type" not in _ref.obs.columns:
            print("\u26a0\ufe0f 参考 atlas 缺少 'cell_type' 列，跳过 scANVI")
            print(f"   可用 obs 列: {list(_ref.obs.columns)}")
        else:
            # 取共同基因（取交集避免训练时基因轴不匹配）
            _common = adata.var_names.intersection(_ref.var_names)
            if len(_common) < 1000:
                print(f"\u26a0\ufe0f 共同基因数不足 ({len(_common)} < 1000)，跳过 scANVI")
            else:
                print(f"  共同基因: {len(_common)}")

                # 构建合并 AnnData：参考细胞保留原标签，目标细胞标记 "Unknown"
                _ref_sub = _ref[:, _common].copy()
                _ref_sub.obs["_scanvi_label"] = _ref_sub.obs["cell_type"].astype(str)

                _q_sub = adata[:, _common].copy()
                _q_sub.obs["_scanvi_label"] = "Unknown"

                # 合并（scvi-tools >= 1.0 单 AnnData 模式）
                _combined = _ref_sub.concatenate(
                    _q_sub, batch_key="_scanvi_batch",
                    batch_categories=["ref", "query"],
                )

                # 设置 SCANVI
                _scvi.model.SCANVI.setup_anndata(
                    _combined, batch_key="_scanvi_batch",
                    labels_key="_scanvi_label",
                )
                _model = _scvi.model.SCANVI(
                    _combined, unlabeled_category="Unknown",
                )

                _max_epochs = 100 if adata.n_obs > 10000 else 50
                _model.train(max_epochs=_max_epochs, early_stopping=True)

                # 获取目标（query）细胞的预测
                _is_query = _combined.obs["_scanvi_batch"] == "query"
                _preds = _model.predict(_combined[_is_query])

                adata.obs[_scANVI_col] = pd.Categorical(
                    _preds["_scanvi_label"].values
                )
                adata.obs[_scANVI_unc_col] = (
                    1.0 - _preds["_scanvi_label"].probabilities.max(axis=1)
                )

                _n_classes = adata.obs[_scANVI_col].nunique()
                print(f"scANVI 完成: {_n_classes} 类 -> obs['{_scANVI_col}']")

                # 记录元数据
                adata.uns[f"scanvi_{ANNOTATION_OUTPUT_VERSION}"] = {
                    "reference_atlas": REFERENCE_ATLAS_PATH,
                    "method": "scANVI",
                    "common_genes": len(_common),
                    "timestamp": datetime.datetime.now().isoformat(),
                }
    except ImportError:
        print("scvi-tools 未安装，跳过 scANVI")
    except Exception as _e:
        print(f"scANVI 运行失败: {_e}")
        import traceback
        traceback.print_exc()
else:
    _reason = (
        "未配置 REFERENCE_ATLAS_PATH" if not REFERENCE_ATLAS_PATH
        else f"{REFERENCE_ATLAS_PATH} 不存在"
    )
    print("=" * 60)
    print(f"scANVI 标签迁移已跳过——原因: {_reason}")
    print("如需启用：在 PARAMS 中设置 REFERENCE_ATLAS_PATH 为有标注的 .h5ad 路径，")
    print("并确保参考数据包含 cell_type obs 列。")
    print("=" * 60)

# 确保列存在（即使跳过也预建空列，保持结构完整）
if _scANVI_col not in adata.obs.columns:
    adata.obs[_scANVI_col] = pd.Categorical([np.nan] * adata.n_obs)
if _scANVI_unc_col not in adata.obs.columns:
    adata.obs[_scANVI_unc_col] = np.nan


## 方法 5（候选，已注释）：CellTypist 预训练分类器

CellTypist 提供多个预训练模型（如 `Immune_All_Low.pkl`、`Developing_Mouse_Brain.pkl` 等）。
**当前注释原因**：PI 的胃/滑膜系统目前没有很好匹配的预训练模型。
当有匹配模型出现时取消下方代码注释即可启用，新列自动进入跨方法比较。

In [ ]:
# === CellTypist（候选——有对应预训练模型时取消注释启用）===
# 前提：pip install celltypist
# 当存在对应组织的 CellTypist 预训练模型时取消注释：
# # import celltypist
# # predictions = celltypist.annotate(
# #     adata, model="Human_Gastric_Atlas.pkl",
# #     majority_voting=True,
# # )
# # adata.obs[f"cell_type_celltypist_{ANNOTATION_OUTPUT_VERSION}"] = (
# #     predictions.predicted_labels["majority_voting"].values
# # )
# # print(f"CellTypist: {adata.obs[f'cell_type_celltypist_{ANNOTATION_OUTPUT_VERSION}'].nunique()} 类")
print(
    "CellTypist 已注释。"
    "当有匹配本组织的预训练模型时取消注释启用。"
)

## 转化状态连续性检测

### 为什么胃粘膜注释不能只给离散标签？

胃上皮细胞存在连续的分化谱系：**主细胞（Chief）→ SPEM → 肠化（IM）**
不是 "A 或 B" 的二元关系。Leiden 聚类虽然把细胞分成了离散的簇，
但某些簇可能落在两个"纯"状态之间的**连续谱**上——这类簇如果强行标注为
单一离散细胞类型，会丢失关键生物学信息。

**检测原理**：对三个已知转化轴，每个轴配一对 declining（下调）和 rising（上调）
标记基因集。如果某个簇对两组 marker 都有明显表达（均值 >0.3），说明该簇
处于**转化中间态**——不应标为终末类型，而应标为 "early SPEM" 或
"Chief→SPEM transitional" 等。

**三个检测轴**（基于胃粘膜已发表文献）：
1. **Chief→SPEM**：PGA3/PGA4/PGC 等主细胞 marker 下降，TFF2/MUC6/CD44 等 SPEM marker 上升
2. **SPEM→IM**：TFF2/MUC6 等 SPEM marker 下降，CDX2/MUC2/VIL1 等肠化 marker 上升
3. **Normal→Atrophy**：ATP4A/ATP4B/GKN1 等壁细胞 marker 下降，TFF1/MUC5AC 等 pit cell marker 上升

检测结果同时存入 `adata.uns["transition_detection_v1"]`，供后续 LLM 证据汇总引用。


In [ ]:
# === 转化状态连续性检测 ===
# 对每簇计算三个转化轴的 declining/rising marker 均值，判断是否处于转化中间态。
# 原理：两个方向的 marker 同时高表达 → 该簇不单纯是某一种终末状态 → 转化中间态
# 为什么不用 pseudotime？pseudotime 需要指定起点，而这里我们关注的是
# "哪几个簇可能不是纯状态"，而非"每个细胞在谱系上的精确位置"。

TRANSITION_AXES = {
    "Chief→SPEM": {
        "declining": ["PGA3", "PGA4", "PGA5", "GIF", "LIPF", "PGC"],
        "rising": ["TFF2", "WFDC2", "MUC6", "CD44", "CFTR", "AQP5"],
    },
    "SPEM→IM": {
        "declining": ["TFF2", "MUC6", "WFDC2"],
        "rising": ["CDX2", "MUC2", "TFF3", "OLFM4", "VIL1", "KRT20"],
    },
    "Normal→Atrophy": {
        "declining": ["ATP4A", "ATP4B", "GKN1", "GKN2"],
        "rising": ["TFF1", "MUC5AC", "CLDN18"],
    },
}

print("===== 转化状态连续性检测 =====\n")
_transition_flags = {}

for axis_name, genes in TRANSITION_AXES.items():
    _declining_present = [g for g in genes["declining"] if g in adata.var_names]
    _rising_present = [g for g in genes["rising"] if g in adata.var_names]

    if len(_declining_present) < 2 or len(_rising_present) < 2:
        print(f"  {axis_name}: 可用标记基因不足（declining={len(_declining_present)}, "
              f"rising={len(_rising_present)}），跳过")
        continue

    print(f"  --- {axis_name} ---")
    print(f"      declining markers: {_declining_present}")
    print(f"      rising markers: {_rising_present}")

    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
        _mask = adata.obs[LEIDEN_COL] == cl

        _dec_expr = adata[_mask][:, _declining_present].X
        _ris_expr = adata[_mask][:, _rising_present].X
        if sp.issparse(_dec_expr):
            _dec_expr = _dec_expr.toarray()
        if sp.issparse(_ris_expr):
            _ris_expr = _ris_expr.toarray()

        _dec_mean = float(_dec_expr.mean())
        _ris_mean = float(_ris_expr.mean())

        # 判断：如果两组 marker 都有明显表达（>0.3），说明处于转化中
        if _dec_mean > 0.3 and _ris_mean > 0.3:
            _ratio = _ris_mean / (_dec_mean + _ris_mean)
            if _ratio < 0.4:
                _stage = "早期"
            elif _ratio < 0.6:
                _stage = "中期"
            else:
                _stage = "晚期"
            print(f"      ⚠️ Cluster {cl}: {axis_name} 转化{_stage} "
                  f"(declining={_dec_mean:.2f}, rising={_ris_mean:.2f}, 进度≈{_ratio:.0%})")
            _transition_flags[str(cl)] = {
                "axis": axis_name,
                "stage": _stage,
                "progress": f"{_ratio:.0%}",
                "declining_mean": f"{_dec_mean:.2f}",
                "rising_mean": f"{_ris_mean:.2f}",
            }
    print()

if _transition_flags:
    print(f"  共 {len(_transition_flags)} 个簇检测到转化信号")
    print("  → 这些簇不应标注为单一离散类型，建议标为 'X→Y transitional' 或 'early SPEM' 等")
    print("  → LLM 判决和 PI 拍板时请参考上述进度信息")
else:
    print("  ✓ 未检测到明显的转化中间态信号")

# 存入 uns 供后续 LLM 证据汇总引用（注意：用 adata.uns 而非依赖 cell 间变量传递）
adata.uns[f"transition_detection_{ANNOTATION_OUTPUT_VERSION}"] = _transition_flags


## 跨方法比较：混淆矩阵热图 + Cohen's kappa + Sankey

对已产生标签的任意两种方法，计算混淆矩阵和 Cohen's kappa，
并用 Sankey 图可视化标签流。

**为什么做跨方法比较？** 这才是多方法并跑的核心价值：
- **一致的方法增强置信度**——两种独立方法给出相同标签 → 可信度高
- **分歧的方法揭示需要 PI 重点复核的簇**——看哪些簇在方法间标签不一致
- **Cohen's kappa 量化一致性**：>0.8 高一致；0.4-0.8 中等一致；<0.4 低一致
- **Sankey 图可视化标签流**——直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"

### 怎么看这些图？
- **混淆矩阵热图**：行=方法A的标签，列=方法B的标签，颜色深浅=归一化比例。
  对角线亮 = 两方法一致；非对角线亮 = 该标签被方法B重新分类。
- **Cohen's kappa 表**：数值越高越一致，<0.4 的方法对建议 PI 重点复核。
- **Sankey 图**：流向矩阵保存在 `results/figures/06_sankey/`，供外部工具绘制。

In [ ]:
# 收集所有已产生的标注列（统一过滤逻辑——与 LLM 证据汇总 cell 保持一致）
# B2：收集所有实际成功跑的标注列（按 ANNOTATION_OUTPUT_VERSION 后缀匹配）
# 有几列就比几列，不强求任何方法
_label_cols = [col for col in sorted(adata.obs.columns)
               if any(kw in col for kw in ("cell_type",))
               and col.endswith(f"_{ANNOTATION_OUTPUT_VERSION}")
               and not any(skip in col for skip in ("proportion", "entropy", "uncertainty", "final"))
               and adata.obs[col].notna().any()]

print(f"可用标注列 ({len(_label_cols)}):")
if len(_label_cols) == 0:
    print("  (无可用标注列——所有注释方法均未产生有效标签)")
elif len(_label_cols) == 1:
    print("  -> 仅一个方法，无交叉比对（不报错）")
if len(_label_cols) == 0:
    print("  (无可用标注列——所有注释方法均未产生有效标签)")
elif len(_label_cols) == 1:
    print("  -> 仅一个方法，无交叉比对（不报错）")
if len(_label_cols) == 0:
    print("  (无可用标注列——所有注释方法均未产生有效标签)")
elif len(_label_cols) == 1:
    print("  -> 仅一个方法，无交叉比对（不报错）")
for col in _label_cols:
    _n = adata.obs[col].nunique()
    print(f"  {col}  ({_n} 类)")
    # 确保是 category 类型
    if adata.obs[col].dtype.name != "category":
        adata.obs[col] = adata.obs[col].astype(str).astype("category")

In [ ]:
# 成对混淆矩阵热图 + Cohen's kappa
# 每个方法对出两张图：(1) 混淆矩阵热图 (2) 打印 kappa 值
from itertools import combinations

if len(_label_cols) >= 2:
    # 多个方法可交叉比对
    _kappa_results = []
    for _a, _b in combinations(_label_cols, 2):
        # 只比较都有有效标签的细胞
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 2:
            continue

        # ---- 混淆矩阵热图 ----
        _cm = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
            normalize="index",
        )
        fig, ax = plt.subplots(
            figsize=(max(6, len(_cm.columns) * 1.3),
                     max(5, len(_cm.index) * 0.8))
        )
        sns.heatmap(_cm, annot=True, fmt=".2f", cmap="Blues",
                    vmin=0, vmax=1, ax=ax,
                    cbar_kws={"label": "proportion (row-normalized)"})
        ax.set_title(f"{_a}  ->  {_b}")
        ax.set_xlabel(_b)
        ax.set_ylabel(_a)
        plt.tight_layout()
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_confusion_{_safe_a}__vs__{_safe_b}.png"
        fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  混淆矩阵热图: {_fname}")
        plt.close("all")

        # ---- Cohen's kappa ----
        _k = annotation_concordance(adata, label_a=_a, label_b=_b)
        _kappa = _k.get("cohen_kappa", np.nan)
        _kappa_results.append({
            "method_a": _a, "method_b": _b,
            "cohen_kappa": round(float(_kappa), 4),
            "n_cells": int(_valid.sum()),
        })
        print(f"  {_a} vs {_b}: kappa={_kappa:.4f} (n={_valid.sum():,})")

    _kappa_df = pd.DataFrame(_kappa_results)
    if len(_kappa_df) > 0:
        print(f"\nCohen's kappa 汇总:")
        try:
            from IPython.display import display as ipy_display
            ipy_display(_kappa_df)
        except ImportError:
            print(_kappa_df)
        # 成对 kappa 写 CSV
        _kappa_df.to_csv("results/figures/06_kappa_pairs.csv", index=False)
        print("  成对 kappa 表已保存: results/figures/06_kappa_pairs.csv")
        adata.uns[f"cross_method_comparison_{ANNOTATION_OUTPUT_VERSION}"] = {
            "label_columns": _label_cols,
            "kappa_csv": "results/figures/06_kappa_pairs.csv",
            "timestamp": datetime.datetime.now().isoformat(),
        }
elif len(_label_cols) == 1:
    print(f"\u2139\ufe0f 仅一个注释方法 ({_label_cols[0]})，无交叉比对（不报错）")
    print("  -> PI 可通过 dotplot 验证标注质量，或多跑一个方法做交叉验证")
else:
    print("\u26a0\ufe0f 无可用标注列——所有注释方法均未产生有效标签")
    print("  -> 请至少启用并成功运行一个注释方法（LLM/marker/scANVI/基因集评分）")

In [ ]:
# Sankey 图——可视化两种方法之间的标签流
# 每个节点 = 一种细胞类型标签，连线宽度 = 细胞数
# 为什么用 Sankey？直观展示"方法 A 的 T_cell 被方法 B 拆成了哪些亚型"
# 流向矩阵保存为 CSV，用 R/plotly 等外部工具绘制（matplotlib.sankey 不稳健）

if len(_label_cols) >= 2:  # B2：有 ≥2 个方法才做 Sankey  # B2：有 ≥2 个方法才做 Sankey  # B2：有 ≥2 个方法才做 Sankey
    for _a, _b in combinations(_label_cols, 2):
        _valid = adata.obs[_a].notna() & adata.obs[_b].notna()
        if _valid.sum() < 10:
            continue
        # 构建流向表（method_a -> method_b）
        _flow = pd.crosstab(
            adata.obs.loc[_valid, _a].astype(str),
            adata.obs.loc[_valid, _b].astype(str),
        )
        _safe_a = _a.replace("/", "_").replace(" ", "_")
        _safe_b = _b.replace("/", "_").replace(" ", "_")
        _fname = f"results/figures/06_sankey/flow_{_safe_a}__vs__{_safe_b}.csv"
        _flow.to_csv(_fname)
        print(f"  流向矩阵已保存: {_fname} (shape={_flow.shape})")

    print(f"\nSankey 流向矩阵已保存至 results/figures/06_sankey/")
    print("（用 R/plotly 等工具绘制 Sankey 图）")
elif len(_label_cols) == 1:
    print(f"Sankey 跳过（仅 {_label_cols[0]} 一个方法，无需流向图）")
else:
    print("Sankey 跳过（无可用标注列）")

## LLM 证据汇总 + 三色分级

LLM 在这里不是"第 N+1 个注释方法"——而是**智能汇总员**。

### 为什么需要 LLM 汇总而不是直接看交叉比对表？

交叉比对（混淆矩阵 + Cohen's kappa）能告诉你"方法 A 和方法 B 在某簇上不一致"，
但不能告诉你**谁更可能对**、**分歧的原因是什么**、**PI 应该怎么裁决**。

LLM 综合以下全部证据，逐簇给出结构化的判决建议：
1. **各方法标签**（marker / mLLMCelltype 共识 / 基因集评分 / scANVI ——如有）
2. **已发表专家注释**（作者原始 `cell_type_original` 列，如有）
3. **Top 10 差异表达基因**（每个簇最特异的 marker）
4. **基因集评分 profile**（该簇对各类细胞标记物的整体评分）
5. **转化状态信号**（该簇是否处于 Chief→SPEM→IM 谱系的中间态）

### 三色输出

| 颜色 | 置信度 | 含义 | PI 动作 |
|------|--------|------|---------|
| GREEN | HIGH | 多方法一致、DEG 与 marker profile 完全吻合 | 可直接接受 |
| YELLOW | MEDIUM | 部分方法有分歧，或 DEG 与预期不完全一致 | 建议查看 dotplot / UMAP 确认 |
| RED | LOW | 多方法严重分歧，或 DEG/marker profile 异常 | PI 必须手动逐基因分析 |

### 怎么看判决结果？

每簇输出一个 JSON，包含：
- `label`：建议的细胞类型标签
- `confidence`：HIGH / MEDIUM / LOW
- `reasoning`：关键判断依据（<=50 字）
- `conflict_analysis`：各方法分歧原因（如无分歧写"一致"）
- `suggestion`：给 PI 的行动建议

结果同时保存到磁盘（`results/figures/06_verdicts/verdict_summary.json`），
方便 PI 在 Jupyter 外查看。


In [ ]:
# === LLM 证据汇总 + 三色分级 ===
# 每簇收集全部证据 -> 结构化 prompt -> 调用 LLM（VERDICT_MODEL_TIER）->
# 解析 JSON 判决 -> 输出汇总表 + 预填 pi_decisions
# 为什么不是"第 N+1 个注释方法"？LLM 在这里不做从头标注——
# 而是综合已有方法的标签、DEG、基因集评分、转化检测，给出"建议标签 + 置信度"。
# 最终判断权在 PI。
#
# 批 5 收口：LLM 调用统一走 llm_config.call_llm_for_annotation()，
# JSON 解析统一走 llm_config.extract_json_from_llm_response()。

print("===== LLM 综合判决（逐簇证据汇总）=====\n")

_verdict_results = {}  # {cluster_id: {label, confidence, reasoning, color, ...}}

# 准备 LLM 配置——从 .env LLM_GROUP 读取
from scrna_integration.llm_config import (
    call_llm_for_annotation,
    extract_json_from_llm_response,
    load_llm_group_config,
)
import json as _json

_cfg = load_llm_group_config(group=LLM_GROUP, project_root=_root)
_can_call_llm = _cfg is not None and _cfg.get("api_key")

if _can_call_llm:
    _verdict_model = _cfg.get("models", {}).get(VERDICT_MODEL_TIER, "")
    _verdict_provider = _cfg.get("provider", "")
    if not _verdict_model:
        print(
            f"VERDICT_MODEL_TIER='{VERDICT_MODEL_TIER}' 在 group 配置中无对应模型，"
            f"跳过证据汇总"
        )
        _can_call_llm = False

if _can_call_llm and _verdict_model:
    # 确保有 rank_genes_groups
    if "rank_genes_06" not in adata.uns:
        sc.tl.rank_genes_groups(
            adata, groupby=LEIDEN_COL, method="wilcoxon",
            n_genes=30, key_added="rank_genes_06",
        )

    # 复用 cell 22 已收集的 _label_cols（如不存在则重新收集）
    if "_label_cols" not in dir() or not _label_cols:
        _label_cols = [col for col in sorted(adata.obs.columns)
                       if any(kw in col for kw in ("cell_type",))
                       and col.endswith(f"_{ANNOTATION_OUTPUT_VERSION}")
                       and not any(skip in col for skip in ("proportion", "entropy", "uncertainty", "final"))
                       and adata.obs[col].notna().any()]

    # 转化检测结果（从 adata.uns 读取，不依赖 cell 间变量）
    _transition_flags = adata.uns.get(f"transition_detection_{ANNOTATION_OUTPUT_VERSION}", {})

    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
        _mask = adata.obs[LEIDEN_COL] == cl

        # ==== 收集证据 ====
        _evidence = f"## Cluster {cl} ({int(_mask.sum())} cells)\n\n"

        # 证据 1: 各方法标签
        _evidence += "### 各方法注释结果\n"
        for col in _label_cols:
            _cl_labels = adata.obs.loc[_mask, col].dropna()
            if len(_cl_labels) > 0:
                _top = _cl_labels.mode().iloc[0]
                _pct = (_cl_labels == _top).mean()
                _evidence += f"- {col}: {_top} ({_pct:.0%})\n"

        # 证据 2: 作者原始注释（如有）
        _ref_col = None
        for cand in ["cell_type_original", "Celltypes_global"]:
            if cand in adata.obs.columns:
                _ref_col = cand
                break
        if _ref_col:
            _ref_labels = adata.obs.loc[_mask, _ref_col].dropna()
            if len(_ref_labels) > 0:
                _ref_top = _ref_labels.mode().iloc[0]
                _ref_pct = (_ref_labels == _ref_top).mean()
                _evidence += f"- **已发表专家注释**({_ref_col}): {_ref_top} ({_ref_pct:.0%})\n"

        # 证据 3: Top DEG
        if "rank_genes_06" in adata.uns:
            _genes = []
            try:
                _df = sc.get.rank_genes_groups_df(adata, group=str(cl), key="rank_genes_06")
                _genes = _df["names"].head(10).tolist()
            except Exception:
                pass
            if _genes:
                _evidence += f"\n### Top 10 DEG\n{', '.join(_genes[:10])}\n"

        # 证据 4: 基因集评分
        _score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
        if _score_cols:
            _evidence += "\n### 基因集评分（簇均值）\n"
            _scores_this = {
                c.replace("score_", ""): f"{adata.obs.loc[_mask, c].mean():.3f}"
                for c in _score_cols
            }
            _sorted_scores = sorted(_scores_this.items(), key=lambda x: -float(x[1]))[:5]
            for name, val in _sorted_scores:
                _evidence += f"- {name}: {val}\n"

        # 证据 5: 转化状态（如检测到）
        if str(cl) in _transition_flags:
            _tf = _transition_flags[str(cl)]
            _evidence += "\n### 转化状态信号\n"
            _evidence += f"- 检测到 {_tf['axis']} 转化（{_tf['stage']}，进度 {_tf['progress']}）\n"
            _evidence += f"- declining markers 均值: {_tf['declining_mean']}, rising markers 均值: {_tf['rising_mean']}\n"
            _evidence += "- 建议：标注为转化中间态而非单一离散类型\n"

        # ==== 构建 prompt ====
        _system = (
            "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱。"
            "请综合以下多维证据，判断该 cluster 的细胞类型。"
        )
        _user = f"""{_evidence}

请以 JSON 格式返回（不要 markdown 围栏）：
{{
  "label": "最可能的细胞类型（具体到亚型）",
  "confidence": "HIGH/MEDIUM/LOW",
  "reasoning": "50字内关键理由",
  "conflict_analysis": "各方法分歧原因（如无分歧写'一致'）",
  "suggestion": "给 PI 的建议（如'可直接接受'或'建议看 dotplot 确认 XX 表达'）"
}}"""

        # ==== 调用 LLM（统一走 llm_config.call_llm_for_annotation）====
        try:
            _raw = call_llm_for_annotation(
                messages=[{"role": "user", "content": _user}],
                model_provider=_verdict_provider,
                model_name=_verdict_model,
                max_tokens=LLM_MAX_TOKENS,
                temperature=0.3,
                system=_system,
                project_root=_root,
                group=LLM_GROUP,
            )

            # 解析 JSON（统一走 llm_config.extract_json_from_llm_response）
            _parsed = extract_json_from_llm_response(_raw)
            if _parsed is None:
                raise ValueError("无法从 LLM 回复中提取 JSON 字典")
            _verdict_results[str(cl)] = _parsed

            _color_map = {"HIGH": "[GREEN]", "MEDIUM": "[YELLOW]", "LOW": "[RED]"}
            _color = _color_map.get(_parsed.get("confidence", ""), "[WHITE]")
            print(f"  {_color} Cluster {cl}: {_parsed.get('label', '?')} "
                  f"({_parsed.get('confidence', '?')}) -- {_parsed.get('reasoning', '')}")

        except Exception as e:
            print(f"  Cluster {cl} 判决失败: {e}")
            _verdict_results[str(cl)] = {
                "label": f"Cluster_{cl} (LLM \u5931\u8d25)",
                "confidence": "LOW",
                "reasoning": f"LLM 调用失败: {e}",
                "conflict_analysis": "",
                "suggestion": "需 PI 手动判断",
            }

    # 保存汇总 JSON
    os.makedirs("results/figures/06_verdicts", exist_ok=True)
    with open("results/figures/06_verdicts/verdict_summary.json", "w") as f:
        _json.dump(_verdict_results, f, ensure_ascii=False, indent=2)
    print("\n判决汇总已保存: results/figures/06_verdicts/verdict_summary.json")

    # 统计三色分布
    _color_counts = {"HIGH": 0, "MEDIUM": 0, "LOW": 0}
    for v in _verdict_results.values():
        _c = v.get("confidence", "")
        if _c in _color_counts:
            _color_counts[_c] += 1
    print(f"三色分布: [GREEN]-HIGH={_color_counts['HIGH']}, "
          f"[YELLOW]-MEDIUM={_color_counts['MEDIUM']}, "
          f"[RED]-LOW={_color_counts['LOW']}")
    print("  -> [GREEN] 簇可直接接受，[YELLOW]/[RED] 请 PI 重点复核")

else:
    print("无可用 LLM 配置（.env 缺少 LLM_GROUP 或 VERDICT_MODEL_TIER 档模型），跳过证据汇总")
    print("  -> fallback: 用 claude -p 手动逐簇分析")
    _verdict_results = {}

# === 写 LLM suggested 列（6b：suggested-only，不落 final） ===
# 把 _verdict_results 的 label 字段落成 cell_type_llm_suggested_{V} obs 列
# 仅对有 LLM 判决结果的簇；无结果簇为 NaN
# 为什么不直接写 final？LLM 是顾问，不是决策者——机器建议必须经 PI 显式确认
_suggested_map = {}
for _cl, _v in _verdict_results.items():
    _label = _v.get("label", "")
    if _label and not _label.startswith("Cluster_"):
        _suggested_map[_cl] = _label

_suggested_col = f"cell_type_llm_suggested_{ANNOTATION_OUTPUT_VERSION}"
if _suggested_map:
    adata.obs[_suggested_col] = (
        adata.obs[LEIDEN_COL].astype(str).map(_suggested_map)
    ).astype("category")
    print(f"\nLLM suggested 列已写入: {_suggested_col}")
    print(f"  覆盖 {len(_suggested_map)}/{adata.obs[LEIDEN_COL].nunique()} 簇")
else:
    adata.obs[_suggested_col] = pd.Categorical([np.nan] * adata.n_obs)
    print(f"\nLLM suggested 列为空（无有效判决），已创建占位列: {_suggested_col}")

# === 初始化 annotation_provenance（6a：每簇记录来源与证据） ===
# provenance 记录每簇的 marker/LLM 建议来源、置信度、推理依据、PI 决定
_prov_key = f"annotation_provenance_{ANNOTATION_OUTPUT_VERSION}"
if _prov_key not in adata.uns:
    adata.uns[_prov_key] = {}
_prov = adata.uns[_prov_key]
for _cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
    _cl_str = str(_cl)
    if _cl_str not in _prov:
        _prov[_cl_str] = {}
    _v = _verdict_results.get(_cl_str, {})
    _prov[_cl_str]["llm_suggested"] = _v.get("label", "") if not (_v.get("label", "") or "").startswith("Cluster_") else ""
    _prov[_cl_str]["llm_confidence"] = _v.get("confidence", "")
    _prov[_cl_str]["llm_reasoning"] = _v.get("reasoning", "")
    _prov[_cl_str]["evidence_refs"] = {
        "has_deg": "rank_genes_06" in adata.uns,
        "has_scores": any(c.startswith("score_") for c in adata.obs.columns),
        "has_transition": str(_cl) in adata.uns.get(f"transition_detection_{ANNOTATION_OUTPUT_VERSION}", {}),
    }
    _prov[_cl_str].setdefault("marker_suggested", "")
    _prov[_cl_str].setdefault("pi_decision", "")
    _prov[_cl_str].setdefault("decision_source", "unresolved")

print(f"annotation_provenance 已初始化: {_prov_key} ({len(_prov)} 簇)")


In [ ]:
# === 注释来源汇总（四版本 provenance，全部 suggested，不落 final）===
# 汇总 marker-based + LLM suggested + 各 method 列 → 补全 annotation_provenance
# 打印每簇 suggested 概览表供 PI 阅读
# 以上均为机器建议，尚未落地为 final

_prov_key = f"annotation_provenance_{ANNOTATION_OUTPUT_VERSION}"
_prov = adata.uns.get(_prov_key, {})

# 补全 marker_suggested（从 marker 注释列读取）
_marker_col = f"cell_type_marker_{ANNOTATION_OUTPUT_VERSION}"
if _marker_col in adata.obs.columns:
    for _cl in adata.obs[LEIDEN_COL].unique():
        _cl_str = str(_cl)
        _mask = adata.obs[LEIDEN_COL] == _cl
        _cl_markers = adata.obs.loc[_mask, _marker_col].dropna()
        if len(_cl_markers) > 0:
            _prov.setdefault(_cl_str, {})["marker_suggested"] = _cl_markers.mode().iloc[0]

print("===== 每簇 suggested 概览（机器建议，未落地为 final）=====")
print()

_label_cols_for_table = [
    c for c in sorted(adata.obs.columns)
    if any(kw in c for kw in ("cell_type",))
    and c.endswith(f"_{ANNOTATION_OUTPUT_VERSION}")
    and "final" not in c
    and not any(skip in c for skip in ("proportion", "entropy", "uncertainty"))
    and adata.obs[c].notna().any()
]

for _cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
    _cl_str = str(_cl)
    _mask = adata.obs[LEIDEN_COL] == _cl
    _n_cells = int(_mask.sum())
    _prov_cl = _prov.get(_cl_str, {})

    _marker_label = _prov_cl.get("marker_suggested", "") or "-"
    _llm_label = _prov_cl.get("llm_suggested", "") or "-"
    _llm_conf = _prov_cl.get("llm_confidence", "") or "-"

    print(f"Cluster {_cl_str} ({_n_cells} cells)")
    print(f"  marker-based:  {_marker_label}")
    print(f"  LLM suggested: {_llm_label}  (confidence: {_llm_conf})")
    for _col in _label_cols_for_table:
        _vals = adata.obs.loc[_mask, _col].dropna()
        if len(_vals) > 0:
            _top = _vals.mode().iloc[0]
            print(f"  {_col}: {_top}")
    _trans = adata.uns.get(f"transition_detection_{ANNOTATION_OUTPUT_VERSION}", {}).get(_cl_str)
    if _trans:
        print(f"  [转化检测] {_trans['axis']}: {_trans['stage']} (progress: {_trans['progress']})")
    print()

print("---")
print("以上均为机器建议（marker-based + LLM + 各 method），尚未落地为 final cell type。")
print("PI 需在下方的决策区逐簇确认或修改后，由闸门 cell 统一生成 cell_type_final。")


## PI 拍板：从 suggested 到 final（决策6：四版本分层）

PI 阅读每簇的 LLM 判决汇总后，手动确认最终标签。
**为什么不让 LLM 自动拍板？** 每个簇的最终标签是科学判断，
LLM 是顾问不是决策者——PI 结合自身领域知识复核后决定。
GREEN 置信度只代表机器认为可靠，不等于 PI 已判断其生物学合理性。

### 四版本注释字段（6a：区分来源）

| 字段 | 含义 | 写入时机 |
|---|---|---|
| `cell_type_marker_{V}` | marker-based 手动标注 | PI 填入 marker_assignments 时 |
| `cell_type_llm_suggested_{V}` | LLM 建议（evidence 汇总） | 证据汇总运行时 |
| `cell_type_pi_confirmed_{V}` | PI 决定（confirmed 或占位） | PI 决策输入 cell |
| `cell_type_final_{V}` | 最终标签 | **仅全簇闸门通过时**（PI_CONFIRMED=True + 全部 resolved） |

> `cell_type_llm_{V}`（mLLMCelltype 共识）与 `cell_type_scanvi_{V}`（scANVI）属于 method 级证据列，不属于四版本最终输出，也永不当 final。

### PI 拍板方式

**方式 1（推荐非 CS 用户）**：在 Excel/WPS 中编辑 CSV，6 列：
`cluster, suggested_label, pi_final_label, decision(accept|modify|unresolved), note, annotation_version`
保存后在上方 PARAMS 中设置 `PI_CONFIRMATION_CSV` 文件路径。
**向后兼容**：旧版 2 列（cluster, label）CSV 视为 decision=modify，可直接使用。

**方式 2（程序员）**：直接在下方 `pi_decisions` Python 字典中填入标签。

### 闸门规则

- **PI_CONFIRMED=False**：永远只写 draft，不写 final（GREEN 也不例外）
- **PI_CONFIRMED=True 但存在 unresolved 簇**：final 不生成，提示缺口
- **PI_CONFIRMED=True 且全簇 resolved + 版本一致**：闸门通过，生成 `cell_type_final_{V}`
- 未确认簇只填占位符 `Cluster_N`（不猜测具体类型）


In [ ]:
# === PI 决策输入（纯 PI 来源，无 LLM 预填）===
# 为什么不再自动预填 GREEN？GREEN 只是 LLM 的机器置信度高，
# 不等于 PI 已判断其生物学合理性。机器建议只作为参考显示，
# pi_decisions 只来自 PI 显式输入（CSV 或 Python dict）。
# 支持两种输入方式：CSV（推荐非 CS 用户）或 Python dict（程序员）

# === 先从 CSV 加载 PI 确认（如果 PARAMS 中配置了 PI_CONFIRMATION_CSV）===
# 新 6 列 schema: cluster, suggested_label, pi_final_label, decision(accept|modify|unresolved), note, annotation_version
# 向后兼容：仅含 cluster, label 两列的旧 CSV 视为 decision=modify（PI 手填即明确决定）
_pi_from_csv = {}
_pi_decision_types = {}  # {cluster: "accept"|"modify"|"unresolved"}
_pi_notes = {}           # {cluster: note}
_pi_csv_version = ""

if PI_CONFIRMATION_CSV and os.path.exists(PI_CONFIRMATION_CSV):
    _pi_df = pd.read_csv(PI_CONFIRMATION_CSV, dtype=str)
    # 检测 schema：6 列新版 or 2 列旧版
    _has_new_schema = "decision" in _pi_df.columns
    if _has_new_schema:
        for _, _row in _pi_df.iterrows():
            _cl = str(_row.get("cluster", "")).strip()
            _label = str(_row.get("pi_final_label", "")).strip()
            _dec = str(_row.get("decision", "")).strip().lower()
            _note = str(_row.get("note", "")).strip()
            _ver = str(_row.get("annotation_version", "")).strip()
            if _cl and _label and _label.lower() not in ("nan", ""):
                _pi_from_csv[_cl] = _label
                _pi_decision_types[_cl] = _dec if _dec in ("accept", "modify", "unresolved") else "modify"
                if _note and _note.lower() != "nan":
                    _pi_notes[_cl] = _note
            elif _cl and _dec == "unresolved":
                _pi_from_csv[_cl] = ""
                _pi_decision_types[_cl] = "unresolved"
                if _note and _note.lower() != "nan":
                    _pi_notes[_cl] = _note
            if _ver and not _pi_csv_version:
                _pi_csv_version = _ver
    else:
        # 旧 2 列兼容：cluster, label
        _cluster_col = None
        _label_col_csv = None
        for c in _pi_df.columns:
            if c.lower() in ("cluster", "cluster_id", "簇"):
                _cluster_col = c
            elif c.lower() in ("label", "cell_type", "标签", "类型"):
                _label_col_csv = c
        if _cluster_col and _label_col_csv:
            for _, _row in _pi_df.iterrows():
                _cl = str(_row.get(_cluster_col, "")).strip()
                _label = str(_row.get(_label_col_csv, "")).strip()
                if _cl and _label and _label.lower() not in ("nan", ""):
                    _pi_from_csv[_cl] = _label
                    _pi_decision_types[_cl] = "modify"  # 旧 CSV = PI 手填 = 明确决定
            print(f"✓ 从 CSV（旧 2 列兼容）读取 PI 决策: {PI_CONFIRMATION_CSV} ({len(_pi_from_csv)} 簇)")
        else:
            print(f"⚠️ CSV 列名不匹配（期望 cluster+label 或新版 6 列），")
            print(f"   实际列名: {list(_pi_df.columns)}")
            PI_CONFIRMATION_CSV = ""

    if _pi_from_csv:
        if not _pi_csv_version:
            _pi_csv_version = ANNOTATION_OUTPUT_VERSION
        print(f"✓ 从 CSV 读取 PI 决策: {PI_CONFIRMATION_CSV} ({len(_pi_from_csv)} 簇)")
        for cl, label in sorted(_pi_from_csv.items(), key=_safe_sort_key):
            _dec = _pi_decision_types.get(cl, "modify")
            print(f"    {cl}: {label}  [{_dec}]")
elif PI_CONFIRMATION_CSV:
    print(f"⚠️ PI_CONFIRMATION_CSV 文件不存在: {PI_CONFIRMATION_CSV}")
    print(f"   → 将使用下方 Python dict")

# === PI 拍板 dict（纯 PI 来源，无 LLM 自动预填）===
pi_decisions = {}
if _pi_from_csv:
    pi_decisions.update(_pi_from_csv)

# PI 也可在下方 Python dict 中补充未在 CSV 中的簇
# 格式: {"cluster_id": "cell_type_label", ...}
# pi_decisions.update({
#     "0": "T cell",
#     "1": "B cell",
# })

# 打印 PI 拍板区——LLM 建议仅供 PI 参考，不自动预填
print()
if "_verdict_results" in dir() and _verdict_results:
    _color_map = {"HIGH": "[GREEN]", "MEDIUM": "[YELLOW]", "LOW": "[RED]"}
    print("===== PI 拍板区（LLM 建议仅供参考，不自动预填）=====")
    print("（请 PI 逐簇确认或修改；未填簇将显示为 Cluster_N 占位，不猜测具体类型）")
    print()
    for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
        _v = _verdict_results.get(str(cl), {})
        _color = _color_map.get(_v.get("confidence", ""), "[WHITE]")
        _llm_label = _v.get("label", "-")
        _llm_reason = _v.get("reasoning", "")
        if str(cl) in pi_decisions and pi_decisions[str(cl)]:
            _source = "CSV" if PI_CONFIRMATION_CSV and str(cl) in _pi_from_csv else "Python dict"
            print(f'  Cluster {cl}: PI="{pi_decisions[str(cl)]}"  (LLM 建议: {_llm_label} {_color}, 来源: {_source})')
        else:
            print(f'  Cluster {cl}: PI=___待填___  (LLM 建议: {_llm_label} {_color}, {_llm_reason})')
else:
    if pi_decisions:
        print("===== PI 拍板区（仅 PI 输入，无 LLM 建议）=====")
        for cl, label in sorted(pi_decisions.items(), key=_safe_sort_key):
            print(f'  Cluster {cl}: PI="{label}"')
    else:
        print("无 LLM 判决结果且无 PI 输入，所有簇待 PI 手工标注")
        for cl in sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key):
            pi_decisions[str(cl)] = ""

# === ACCEPT_ALL_SUGGESTED 批量操作 ===
# 仅当 PI_CONFIRMED=True 且 ACCEPT_ALL_SUGGESTED=True 时，
# 才从 suggested 生成逐簇 decision=accept 审计记录
if PI_CONFIRMED and ACCEPT_ALL_SUGGESTED and "_verdict_results" in dir() and _verdict_results:
    _accepted_count = 0
    for cl in adata.obs[LEIDEN_COL].unique():
        _cl_str = str(cl)
        _v = _verdict_results.get(_cl_str, {})
        _sugg = _v.get("label", "")
        if _sugg and not _sugg.startswith("Cluster_") and (_cl_str not in pi_decisions or not pi_decisions[_cl_str]):
            pi_decisions[_cl_str] = _sugg
            _pi_decision_types[_cl_str] = "accept"
            _accepted_count += 1
    if _accepted_count > 0:
        print(f"\nACCEPT_ALL_SUGGESTED: 已批量接受 {_accepted_count} 个簇的 LLM 建议")
        print("  审计记录: 每簇 decision_source='accept'")

# === 写 cell_type_pi_confirmed（PI 决定层，不是 final）===
# PI 已决簇填其标签，未决簇填 Cluster_N 占位（6d：占位不是猜测的类型）
_pi_confirmed_col = f"cell_type_pi_confirmed_{ANNOTATION_OUTPUT_VERSION}"
_pi_map = {}
for _cl in adata.obs[LEIDEN_COL].unique():
    _cl_str = str(_cl)
    if _cl_str in pi_decisions and pi_decisions[_cl_str]:
        _pi_map[_cl_str] = pi_decisions[_cl_str]
    else:
        _pi_map[_cl_str] = f"Cluster_{_cl_str}"

adata.obs[_pi_confirmed_col] = (
    adata.obs[LEIDEN_COL].astype(str).map(_pi_map)
).astype("category")
print(f"\nPI confirmed 列已写入: {_pi_confirmed_col}")
_n_confirmed = sum(1 for _cl in adata.obs[LEIDEN_COL].unique()
                   if str(_cl) in pi_decisions and pi_decisions[str(_cl)])
_n_total = adata.obs[LEIDEN_COL].nunique()
_n_placeholder = _n_total - _n_confirmed
print(f"  已确认: {_n_confirmed}/{_n_total} 簇")
if _n_placeholder > 0:
    print(f"  占位符: {_n_placeholder} 簇（填为 Cluster_N，非科学标注）")

# === 补全 annotation_provenance 的 PI 决策信息 ===
_prov_key = f"annotation_provenance_{ANNOTATION_OUTPUT_VERSION}"
_prov = adata.uns.setdefault(_prov_key, {})
for _cl in adata.obs[LEIDEN_COL].unique():
    _cl_str = str(_cl)
    _prov.setdefault(_cl_str, {})
    if _cl_str in pi_decisions and pi_decisions[_cl_str]:
        _prov[_cl_str]["pi_decision"] = pi_decisions[_cl_str]
        _prov[_cl_str]["decision_source"] = _pi_decision_types.get(_cl_str, "modify")
    else:
        _prov[_cl_str]["pi_decision"] = ""
        _prov[_cl_str]["decision_source"] = "unresolved"
    if _cl_str in _pi_notes:
        _prov[_cl_str]["pi_note"] = _pi_notes[_cl_str]
adata.uns[_prov_key] = _prov


In [ ]:
# === PI-final 闸门（全簇确认才生成 final）===
# 决策6 (6b/6c/6d)：final cell type 必须经 PI 显式确认，机器建议不自动落地
# 闸门逻辑：
#   1. 逐簇检查 decision_source —— 全部非 unresolved
#   2. 版本一致性 —— annotation_version == ANNOTATION_OUTPUT_VERSION
#   3. PI_CONFIRMED 总开关为 True
#   全部满足 → final_gate_passed = True → 写 cell_type_final 列
#   任一不满足 → final_gate_passed = False → 仅保存 draft

_prov_key = f"annotation_provenance_{ANNOTATION_OUTPUT_VERSION}"
_prov = adata.uns.get(_prov_key, {})

_all_clusters = sorted(adata.obs[LEIDEN_COL].unique(), key=_safe_sort_key)
_unresolved_clusters = []
_decision_counts = {"accept": 0, "modify": 0, "unresolved": 0}
_all_clusters_resolved = True

for _cl in _all_clusters:
    _cl_str = str(_cl)
    _ds = _prov.get(_cl_str, {}).get("decision_source", "unresolved")
    _decision_counts[_ds] = _decision_counts.get(_ds, 0) + 1
    if _ds == "unresolved":
        _all_clusters_resolved = False
        _unresolved_clusters.append(_cl_str)

# 版本一致性检查
if PI_CONFIRMATION_CSV and _pi_csv_version:
    _version_ok = (_pi_csv_version == ANNOTATION_OUTPUT_VERSION)
else:
    _version_ok = True
    _pi_csv_version = ANNOTATION_OUTPUT_VERSION

_no_unresolved = len(_unresolved_clusters) == 0

# 闸门判定
final_gate_passed = bool(
    PI_CONFIRMED
    and _all_clusters_resolved
    and _no_unresolved
    and _version_ok
)

print("===== PI-final 闸门 =====")
print(f"  PI_CONFIRMED:          {PI_CONFIRMED}")
print(f"  all_clusters_resolved: {_all_clusters_resolved}")
print(f"  no_unresolved:         {_no_unresolved}")
print(f"  version_ok:            {_version_ok}")
print(f"  decision_counts:       {_decision_counts}")
print(f"  => final_gate_passed:  {final_gate_passed}")
print()

_final_col = f"cell_type_final_{ANNOTATION_OUTPUT_VERSION}"
_notes_key = f"cell_type_final_{ANNOTATION_OUTPUT_VERSION}_notes"

if final_gate_passed:
    _final_map = {}
    for _cl in _all_clusters:
        _cl_str = str(_cl)
        _pi_label = _prov.get(_cl_str, {}).get("pi_decision", "")
        if _pi_label:
            _final_map[_cl_str] = _pi_label
        else:
            _final_map[_cl_str] = f"Cluster_{_cl_str}"

    adata.obs[_final_col] = (
        adata.obs[LEIDEN_COL].astype(str).map(_final_map)
    ).astype("category")
    print(f"✓ cell_type_final 已写入: {_final_col}")
    print(f"  标签种类: {adata.obs[_final_col].nunique()}")

    import datetime, hashlib
    _confirmation_hash = ""
    if PI_CONFIRMATION_CSV and os.path.exists(PI_CONFIRMATION_CSV):
        with open(PI_CONFIRMATION_CSV, "rb") as _fh:
            _confirmation_hash = hashlib.sha256(_fh.read()).hexdigest()

    adata.uns[_notes_key] = {
        "status": "final",
        "confirmed_at": datetime.datetime.now().isoformat(),
        "annotation_version": ANNOTATION_OUTPUT_VERSION,
        "decision_source_counts": _decision_counts,
        "all_clusters_resolved": True,
        "confirmation_file": PI_CONFIRMATION_CSV if PI_CONFIRMATION_CSV else "(Python dict)",
        "confirmation_file_sha256": _confirmation_hash,
        "method_basis": "PI confirmed final labels from suggested candidates",
        "rationale": "所有簇经 PI 显式确认后统一生成 final；机器建议未自动落地",
    }
    print(f"  notes 已写入: {_notes_key}")
else:
    print("✗ cell_type_final 未创建——闸门未通过")
    _reasons = []
    if not PI_CONFIRMED:
        _reasons.append("PI_CONFIRMED=False")
    if not _all_clusters_resolved:
        _reasons.append(f"未决簇: {_unresolved_clusters}")
    if not _version_ok:
        _reasons.append(f"版本不一致: CSV={_pi_csv_version}, notebook={ANNOTATION_OUTPUT_VERSION}")
    for _r in _reasons:
        print(f"  - {_r}")
    print()
    print("  → 仅保存 draft（suggested + pi_confirmed），等待 PI 全簇确认。")
    print("  → PI 操作：填写 PI_CONFIRMATION_CSV 或修改 pi_decisions，")
    print("    确保所有簇 decision 不为 unresolved，然后设置 PI_CONFIRMED=True 重新运行。")

    adata.uns[_notes_key] = {
        "status": "draft-suggested-only",
        "method_basis": "draft: machine suggestions only, PI confirmation pending",
        "annotation_version": ANNOTATION_OUTPUT_VERSION,
        "unresolved_clusters": _unresolved_clusters,
        "decision_source_counts": _decision_counts,
        "pi_confirmed": PI_CONFIRMED,
        "note": "machine suggestions are not final; PI confirmation required for all clusters",
    }
    print(f"  draft notes 已写入: {_notes_key}")

# 记录闸门状态到 uns（供 checkpoint cell 和下游读取）
adata.uns["annotation_gate"] = {
    "pi_confirmed": PI_CONFIRMED,
    "final_gate_passed": final_gate_passed,
    "all_clusters_resolved": _all_clusters_resolved,
    "unresolved_clusters": _unresolved_clusters,
    "decision_source_counts": _decision_counts,
    "annotation_version": ANNOTATION_OUTPUT_VERSION,
}
print("annotation_gate 已记录到 adata.uns")


In [ ]:
# 记录 PI 拍板的元数据——按 final_gate_passed 写真实基线
# 删除旧版硬编码字符串——机器建议未自动落地时不能声称 PI 已审阅
_notes_key = f"cell_type_final_{ANNOTATION_OUTPUT_VERSION}_notes"
_notes = adata.uns.get(_notes_key, {})
_prov_key = f"annotation_provenance_{ANNOTATION_OUTPUT_VERSION}"
_prov = adata.uns.get(_prov_key, {})

# 补充方法列表
_label_cols_final = [c for c in sorted(adata.obs.columns)
                     if any(kw in c for kw in ("cell_type",))
                     and c.endswith(f"_{ANNOTATION_OUTPUT_VERSION}")
                     and not any(skip in c for skip in ("proportion", "entropy", "uncertainty"))
                     and adata.obs[c].notna().any()]

_notes.setdefault("leiden_resolution_used", LEIDEN_COL)
_notes.setdefault("available_methods", _label_cols_final if "_label_cols_final" in dir() else [])
_notes.setdefault("timestamp", datetime.datetime.now().isoformat())

adata.uns[_notes_key] = _notes
print(f"PI 拍板元数据已更新: adata.uns['{_notes_key}']")


In [ ]:
# 运行追踪字段——Stage 06 当前仅保存待审查 draft
adata.uns["stage"] = "06_annotated"
adata.uns["version"] = ANNOTATION_OUTPUT_VERSION
adata.uns["run_id"] = RUN_ID
adata.uns["upstream"] = str(UPSTREAM_CHECKPOINT)
adata.uns["upstream_inputs"] = {"stage05": upstream_input}
adata.uns["status"] = "NEEDS_REVIEW"

# annotation_gate 由闸门 cell 写入，此处不重复设置
if "annotation_gate" not in adata.uns:
    adata.uns["annotation_gate"] = {
        "pi_confirmed": PI_CONFIRMED,
        "final_gate_passed": final_gate_passed,
        "annotation_version": ANNOTATION_OUTPUT_VERSION,
    }

# 记录 06 运行元数据
adata.uns[f"06_annotated_{ANNOTATION_OUTPUT_VERSION}"] = {
    "upstream_run_id": UPSTREAM_RUN_ID,
    "leiden_col": LEIDEN_COL,
    "marker_csv": MARKER_CSV,
    "methods_run": [
        "marker_dotplot",
        "mllmcelltype_consensus",
        "gene_set_scoring",
        "transition_detection",
        "llm_evidence_verdict",
        "pi_decision_input",
        "pi_final_gate",
    ],
    "scANVI_skipped": not bool(REFERENCE_ATLAS_PATH),
    "timestamp": datetime.datetime.now().isoformat(),
}
print("06 运行元数据已记录")


In [ ]:
# 写入前自检——守卫 adata.X 不变
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("自检通过: X 是 sparse CSR float32")

In [ ]:
# === Stage 06 draft checkpoint：PI-final 闸门接线 ===
# needs_review 来源改为 final_gate_passed（不再硬编码 True）
# 闸门通过 → stage_status=SUCCESS（可被外部步骤提升为 promoted）
# 闸门未通过 → stage_status=NEEDS_REVIEW（仅 draft，不可提升）
# 状态提升不在此 cell 调用——仍由外部 operator 步骤执行
cluster_column_present = LEIDEN_COL in adata.obs.columns
cluster_labels_complete = cluster_column_present and bool(adata.obs[LEIDEN_COL].notna().all())
cluster_labels_nonempty = cluster_labels_complete and bool(
    adata.obs[LEIDEN_COL].astype(str).str.strip().ne("").all()
)
output_filename_valid = (
    isinstance(OUTPUT_FILENAME, str) and bool(OUTPUT_FILENAME.strip())
    and OUTPUT_FILENAME not in {".", "..", "manifest.json"}
    and not Path(OUTPUT_FILENAME).is_absolute()
    and Path(OUTPUT_FILENAME).name == OUTPUT_FILENAME
    and "/" not in OUTPUT_FILENAME and "\\" not in OUTPUT_FILENAME
    and re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*\.h5ad", OUTPUT_FILENAME) is not None
    and not any(ord(char) < 32 or ord(char) == 127 for char in OUTPUT_FILENAME)
)
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "cluster_column_present": cluster_column_present,
    "cluster_labels_complete": cluster_labels_complete,
    "cluster_labels_nonempty": cluster_labels_nonempty,
    "output_filename_valid": output_filename_valid,
}
# needs_review 由 final_gate_passed 决定（不再硬编码 True）
needs_review = not final_gate_passed
stage_status = determine_stage_status(
    {}, hard_postconditions, needs_review=needs_review, allow_no_required_methods=True
)
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("UPSTREAM_CHECKPOINT",), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, ("anndata", "scanpy", "numpy", "pandas", "scipy", "scikit-learn", "seaborn", "mllmcelltype", "scvi-tools", "torch")
)
manifest_payload = {
    "run_id": RUN_ID, "stage": "06_annotated", "stage_status": stage_status.value,
    "inputs": [upstream_input], "effective_parameters": effective_parameters,
    "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions, "cluster_key": LEIDEN_COL,
    # 追加 annotation_gate（红线：仅追加 manifest_payload 键，不改既有门禁语义）
    "annotation_gate": adata.uns.get("annotation_gate", {}),
}
run_paths = prepare_run(RUN_ROOT, RUN_ID)
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 06 FAILED: {hard_postconditions}")
adata.uns["stage"] = "06_annotated"
adata.uns["status"] = stage_status.value
adata.uns["run_id"] = RUN_ID
adata.uns["upstream"] = str(UPSTREAM_CHECKPOINT)
adata.uns["upstream_inputs"] = {"stage05": upstream_input}
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
draft_output_path = validate_checkpoint(run_paths.manifest_path)
print(f"已保存待审查 draft: {draft_output_path}")
if final_gate_passed:
    print("stage_status: SUCCESS — PI-final 闸门已通过，所有簇已确认，可执行 promote")
else:
    _unresolved = adata.uns.get("annotation_gate", {}).get("unresolved_clusters", [])
    if _unresolved:
        print(f"stage_status: NEEDS_REVIEW — 未决簇: {_unresolved}；未执行 PI-final 确认")
    else:
        print("stage_status: NEEDS_REVIEW — PI_CONFIRMED=False 或有未决簇；未执行 PI-final 确认或 promotion")


### Stage 06 Verdict

本 stage 完成后应确认：
- [ ] PI 已审阅 LLM 三色判决（GREEN 接受 / YELLOW+RED 已手动决策）
- [ ] cell_type_final_v1 所有簇已填入标签
- [ ] 如需精细注释某 compartment -> 转 06c subset


In [ ]:
# 跨 stage 边界释放内存
del adata
gc.collect()
print("内存已释放")